<a href="https://colab.research.google.com/github/106124113/Tail-Risk-VaR-Expected-Shortfall/blob/main/Tailrisk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


Readed the files from google drive

In [ ]:
import pandas as pd
auto=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NIFTY_AUTO.csv')
fin=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NIFTY_FINANCIALSERVICES.csv')
it=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NIFTY_IT.csv')



calculated logreturns of all sectors


In [ ]:
import pandas as pd
import numpy as np
auto['log_return_auto']=np.log(auto['NIFTY AUTO']/auto['NIFTY AUTO'].shift(1))
fin['log_return_fin']=np.log(fin['NIFTY FIN SERVICE']/fin['NIFTY FIN SERVICE'].shift(1))
it['log_return_it']=np.log(it['NIFTY IT']/it['NIFTY IT'].shift(1))
auto.dropna(inplace=True)
fin.dropna(inplace=True)
it.dropna(inplace=True)
print(auto.head())
print(fin.head())
print(it.head())

              DateTime  NIFTY AUTO  log_return_auto
1  2020-12-02 00:00:00     9093.70         0.011786
2  2020-12-03 00:00:00     9245.30         0.016533
3  2020-12-04 00:00:00     9302.20         0.006136
4  2020-12-07 00:00:00     9311.10         0.000956
5  2020-12-08 00:00:00     9297.85        -0.001424
              DateTime  NIFTY FIN SERVICE  log_return_fin
1  2020-12-02 00:00:00           14187.50       -0.011531
2  2020-12-03 00:00:00           14138.75       -0.003442
3  2020-12-04 00:00:00           14286.45        0.010392
4  2020-12-07 00:00:00           14360.25        0.005152
5  2020-12-08 00:00:00           14384.70        0.001701
              DateTime  NIFTY IT  log_return_it
1  2020-12-02 00:00:00  22319.00       0.006660
2  2020-12-03 00:00:00  22202.20      -0.005247
3  2020-12-04 00:00:00  22309.80       0.004835
4  2020-12-07 00:00:00  22440.85       0.005857
5  2020-12-08 00:00:00  22617.35       0.007834


HISTORICAL SIMULATION

In [ ]:
import numpy as np
import pandas as pd

def compute_var_es_series(returns, window=250):
    VaR_95 = []
    VaR_99 = []
    ES_95  = []
    ES_99  = []

    for i in range(len(returns)):
        if i < window:
            VaR_95.append(np.nan)
            VaR_99.append(np.nan)
            ES_95.append(np.nan)
            ES_99.append(np.nan)
        else:
            past = returns.iloc[i-window:i]

            var95 = np.percentile(past, 5)
            var99 = np.percentile(past, 1)

            es95 = past[past <= var95].mean()
            es99 = past[past <= var99].mean()

            VaR_95.append(var95)
            VaR_99.append(var99)
            ES_95.append(es95)
            ES_99.append(es99)

    return (
        pd.Series(VaR_95, index=returns.index),
        pd.Series(VaR_99, index=returns.index),
        pd.Series(ES_95,  index=returns.index),
        pd.Series(ES_99,  index=returns.index)
    )


In [ ]:
var_hist_95_auto, var_hist_99_auto, es_hist_95_auto, es_hist_99_auto = \
    compute_var_es_series(auto['log_return_auto'])

var_hist_95_fin, var_hist_99_fin, es_hist_95_fin, es_hist_99_fin = \
    compute_var_es_series(fin['log_return_fin'])

var_hist_95_it, var_hist_99_it, es_hist_95_it, es_hist_99_it = \
    compute_var_es_series(it['log_return_it'])


by parametric gaussian distribution

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def rolling_parametric_var_es(returns, window=250, alpha=0.95):
    VaR = []
    ES  = []

    z = norm.ppf(1 - alpha)

    for i in range(len(returns)):
        if i < window:
            VaR.append(np.nan)
            ES.append(np.nan)
        else:
            past = returns.iloc[i-window:i]

            mu = past.mean()
            sigma = past.std()

            var = mu + sigma * z
            es  = mu - sigma * (norm.pdf(z) / (1 - alpha))

            VaR.append(var)
            ES.append(es)

    return (
        pd.Series(VaR, index=returns.index),
        pd.Series(ES,  index=returns.index)
    )


In [ ]:
# AUTO
var_param_95_auto, es_param_95_auto = rolling_parametric_var_es(
    auto['log_return_auto'], alpha=0.95)

var_param_99_auto, es_param_99_auto = rolling_parametric_var_es(
    auto['log_return_auto'], alpha=0.99)

# IT
var_param_95_it, es_param_95_it = rolling_parametric_var_es(
    it['log_return_it'], alpha=0.95)

var_param_99_it, es_param_99_it = rolling_parametric_var_es(
    it['log_return_it'], alpha=0.99)

# FIN
var_param_95_fin, es_param_95_fin = rolling_parametric_var_es(
    fin['log_return_fin'], alpha=0.95)

var_param_99_fin, es_param_99_fin = rolling_parametric_var_es(
    fin['log_return_fin'], alpha=0.99)


Monte carlo method

In [ ]:
!pip install arch tqdm --quiet

import numpy as np
import pandas as pd
from arch import arch_model
from scipy.stats import t, chi2
from tqdm import tqdm


# ---------------------------------------------------
# Rolling full refit Monte Carlo t-GARCH VaR & ES
# ---------------------------------------------------
def rolling_garch_mc_var_es(returns, window=250, sims=5000, alpha=0.95, dist='t', mean='Constant', progress=True):
    if not isinstance(returns, pd.Series):
        returns = pd.Series(returns)
    n = len(returns)
    # prepare output series (NaN for early days)
    var_series = pd.Series(index=returns.index, dtype=float)
    es_series  = pd.Series(index=returns.index, dtype=float)
    sigma_series = pd.Series(index=returns.index, dtype=float)
    nu_series = pd.Series(index=returns.index, dtype=float)
    # loop through forecast dates
    rng = range(window, n)
    if progress:
        rng = tqdm(rng, desc="Rolling GARCH MC")
    for i in rng:
        train = returns.iloc[i - window: i]   # last 'window' days
        # fit GARCH(1,1)
        try:
            model = arch_model(train, mean=mean, lags=1 if mean=='AR' else 0, vol='Garch', p=1, q=1, dist=dist)
            res = model.fit(disp='off')           # silent
        except Exception as e:
            # on rare fit errors, skip day
            print(f"fit error at index {returns.index[i]}: {e}")
            var_series.iloc[i] = np.nan
            es_series.iloc[i] = np.nan
            continue
        # forecast one-step ahead mean & variance
        forecast = res.forecast(horizon=1)
        try:
            mu = forecast.mean.values[-1, 0]
        except:
            # fallback to sample mean if forecast mean missing
            mu = train.mean()
        sigma_t1 = np.sqrt(forecast.variance.values[-1, 0])
        # simulate shocks
        if dist.lower() == 't':
            nu = float(res.params.get('nu', 8.0))
            eps = t.rvs(df=nu, size=sims)
            nu_series.iloc[i] = nu
        else:
            eps = np.random.normal(0, 1, sims)
            nu_series.iloc[i] = np.nan
        # simulated returns
        simulated = mu + sigma_t1 * eps
        # VaR & ES (left tail)
        var_i = np.percentile(simulated, (1 - alpha) * 100)
        tail = simulated[simulated <= var_i]
        es_i = tail.mean() if tail.size > 0 else np.nan
        var_series.iloc[i] = var_i
        es_series.iloc[i]  = es_i
        sigma_series.iloc[i] = sigma_t1
    return var_series, es_series, sigma_series, nu_series

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 14.8 MB/s eta 0:00:00


In [ ]:
# ---------------------------------------------------
# AUTO sector
# ---------------------------------------------------
var_mc_95_auto, es_mc_95_auto, sigma_mc_auto, nu_mc_auto = rolling_garch_mc_var_es(
    auto['log_return_auto'], window=250, sims=5000, alpha=0.95, dist='t', mean='Constant', progress=True
)

var_mc_99_auto, es_mc_99_auto, sigma_mc_auto_99, nu_mc_auto_99 = rolling_garch_mc_var_es(
    auto['log_return_auto'], window=250, sims=5000, alpha=0.99, dist='t', mean='Constant', progress=True
)

# ---------------------------------------------------
# IT sector
# ---------------------------------------------------
var_mc_95_it, es_mc_95_it, sigma_mc_it, nu_mc_it = rolling_garch_mc_var_es(
    it['log_return_it'], window=250, sims=5000, alpha=0.95, dist='t', mean='Constant', progress=True
)

var_mc_99_it, es_mc_99_it, sigma_mc_it_99, nu_mc_it_99 = rolling_garch_mc_var_es(
    it['log_return_it'], window=250, sims=5000, alpha=0.99, dist='t', mean='Constant', progress=True
)

# ---------------------------------------------------
# FIN sector
# ---------------------------------------------------
var_mc_95_fin, es_mc_95_fin, sigma_mc_fin, nu_mc_fin = rolling_garch_mc_var_es(
    fin['log_return_fin'], window=250, sims=5000, alpha=0.95, dist='t', mean='Constant', progress=True
)
var_mc_99_fin, es_mc_99_fin, sigma_mc_fin_99, nu_mc_fin_99 = rolling_garch_mc_var_es(
    fin['log_return_fin'], window=250, sims=5000, alpha=0.99, dist='t', mean='Constant', progress=True
)

Streaming output truncated to the last 5000 lines.
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
Rolling GARCH MC:  47%|████▋     | 459/987 [00:49<00:55,  9.44it/s]/usr/local/lib/python3.12/dist-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 5.769e-05. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
/usr/local/lib/python3.12/dist-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 5.759e-05. Parameter
estimation work bett

In [ ]:
import numpy as np
from scipy.stats import chi2

def kupiec_test(returns, var_series, alpha=0.95):
    violations = (returns < var_series).astype(int).to_numpy()
    N = len(violations)
    x = violations.sum()
    pi_hat = x / N
    epsilon = 1e-10  # avoid log(0)
    LR = -2 * (
        np.log(((1-alpha)**(N-x) * (alpha**x)) + epsilon) -
        np.log(((1-pi_hat)**(N-x) * (pi_hat**x)) + epsilon)
    )
    p_value = 1 - chi2.cdf(LR, df=1)
    return LR, p_value, x, N


def christoffersen_test(returns, var_series):
    violations = (returns < var_series).astype(int)
    N = len(violations)

    n00 = n01 = n10 = n11 = 0
    for i in range(1, N):
        if violations.iloc[i-1] == 0 and violations.iloc[i] == 0:
            n00 += 1
        elif violations.iloc[i-1] == 0 and violations.iloc[i] == 1:
            n01 += 1
        elif violations.iloc[i-1] == 1 and violations.iloc[i] == 0:
            n10 += 1
        elif violations.iloc[i-1] == 1 and violations.iloc[i] == 1:
            n11 += 1

    pi0 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi1 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11) if (n00 + n01 + n10 + n11) > 0 else 0

    L0 = ((1-pi)**(n00+n10)) * (pi**(n01+n11)) if pi not in [0,1] else 1e-10
    L1 = ((1-pi0)**n00) * (pi0**n01) * ((1-pi1)**n10) * (pi1**n11)
    L1 = L1 if L1 != 0 else 1e-10

    LR = -2 * np.log(L0 / L1)
    p_value = 1 - chi2.cdf(LR, df=2)
    return LR, p_value


In [ ]:
sectors = {
    'AUTO': auto['log_return_auto'],
    'IT': it['log_return_it'],
    'FIN': fin['log_return_fin']
}

alphas = [0.95, 0.99]

var_dict = {
    'Historical': {
        0.95: {'AUTO': var_hist_95_auto, 'IT': var_hist_95_it, 'FIN': var_hist_95_fin},
        0.99: {'AUTO': var_hist_99_auto, 'IT': var_hist_99_it, 'FIN': var_hist_99_fin},
    },
    'Parametric': {
        0.95: {'AUTO': var_param_95_auto, 'IT': var_param_95_it, 'FIN': var_param_95_fin},
        0.99: {'AUTO': var_param_99_auto, 'IT': var_param_99_it, 'FIN': var_param_99_fin},
    },
    'MonteCarlo': {
        0.95: {'AUTO': var_mc_95_auto, 'IT': var_mc_95_it, 'FIN': var_mc_95_fin},
        0.99: {'AUTO': var_mc_99_auto, 'IT': var_mc_99_it, 'FIN': var_mc_99_fin},
    }
}


In [ ]:
import pandas as pd

results = []

for method, alpha_dict in var_dict.items():
    for alpha, sector_vars in alpha_dict.items():
        for sector, var_series in sector_vars.items():
            # Kupiec test
            LR_kup, p_kup, x, N = kupiec_test(sectors[sector], var_series, alpha=alpha)
            # Christoffersen test
            LR_cc, p_cc = christoffersen_test(sectors[sector], var_series)

            results.append({
                'Method': method,
                'Sector': sector,
                'Alpha': alpha,
                'Kupiec_LR': LR_kup,
                'Kupiec_pval': p_kup,
                'Violations': f"{x}/{N}",
                'Christoffersen_LR': LR_cc,
                'Christoffersen_pval': p_cc
            })

results_df = pd.DataFrame(results)
print(results_df)


        Method Sector  Alpha     Kupiec_LR  Kupiec_pval Violations  \
0   Historical   AUTO   0.95 -0.000000e+00     1.000000    52/1237   
1   Historical     IT   0.95 -0.000000e+00     1.000000    53/1237   
2   Historical    FIN   0.95 -0.000000e+00     1.000000    40/1237   
3   Historical   AUTO   0.99 -0.000000e+00     1.000000    14/1237   
4   Historical     IT   0.99 -0.000000e+00     1.000000    17/1237   
5   Historical    FIN   0.99  1.492140e-13     1.000000     9/1237   
6   Parametric   AUTO   0.95 -0.000000e+00     1.000000    53/1237   
7   Parametric     IT   0.95 -0.000000e+00     1.000000    53/1237   
8   Parametric    FIN   0.95 -0.000000e+00     1.000000    41/1237   
9   Parametric   AUTO   0.99 -0.000000e+00     1.000000    19/1237   
10  Parametric     IT   0.99 -0.000000e+00     1.000000    20/1237   
11  Parametric    FIN   0.99 -0.000000e+00     1.000000    14/1237   
12  MonteCarlo   AUTO   0.95 -0.000000e+00     1.000000    23/1237   
13  MonteCarlo     I

combined the daily log returns of the NIFTY AUTO, NIFTY IT, and NIFTY FIN indices into a single  to represent a portfolio.

In [ ]:
returns_df = pd.DataFrame({
    'AUTO': auto['log_return_auto'],
    'IT': it['log_return_it'],
    'FIN': fin['log_return_fin']
}).dropna()

equal-weighted portfolio,

In [ ]:
weights = np.array([1/3, 1/3, 1/3])
mu = returns_df.mean().values
cov = returns_df.cov().values

calculated portfolio var and es

In [ ]:
portfolio_mu = weights @ mu
portfolio_sigma = np.sqrt(weights.T @ cov @ weights)
def portfolio_var_es(mu_p, sigma_p, alpha=0.95):
    z = norm.ppf(1 - alpha)
    VaR = mu_p + z * sigma_p
    ES = mu_p - sigma_p * (norm.pdf(z) / (1 - alpha))
    return VaR, ES

In [ ]:
var_p_95, es_p_95 = portfolio_var_es(portfolio_mu, portfolio_sigma, 0.95)
var_p_99, es_p_99 = portfolio_var_es(portfolio_mu, portfolio_sigma, 0.99)

In [ ]:
print("Portfolio Parametric VaR & ES (with correlations)")
print("VaR 95%:", var_p_95)
print("ES  95%:", es_p_95)
print("VaR 99%:", var_p_99)
print("ES  99%:", es_p_99)


Portfolio Parametric VaR & ES (with correlations)
VaR 95%: -0.015328220917102484
ES  95%: -0.019380523657620084
VaR 99%: -0.02193719622178035
ES  99%: -0.025223444526708806


In [ ]:
print("\nCorrelation Matrix of Sector Returns")
print(returns_df.corr())



Correlation Matrix of Sector Returns
          AUTO        IT       FIN
AUTO  1.000000  0.395200  0.612631
IT    0.395200  1.000000  0.391445
FIN   0.612631  0.391445  1.000000


compared the portfolio VaR with individual sector VaRs

In [ ]:
print("\nIndividual Sector VaR 95%")
print("AUTO:", var_hist_95_auto)
print("IT  :", var_hist_95_it)
print("FIN :", var_hist_95_fin)

print("\nPortfolio VaR 95%:", var_p_95)


Individual Sector VaR 95%
AUTO: 1            NaN
2            NaN
3            NaN
4            NaN
5            NaN
          ...   
1233   -0.016547
1234   -0.016547
1235   -0.016547
1236   -0.016547
1237   -0.016547
Length: 1237, dtype: float64
IT  : 1           NaN
2           NaN
3           NaN
4           NaN
5           NaN
         ...   
1233   -0.02436
1234   -0.02436
1235   -0.02436
1236   -0.02436
1237   -0.02436
Length: 1237, dtype: float64
FIN : 1            NaN
2            NaN
3            NaN
4            NaN
5            NaN
          ...   
1233   -0.012248
1234   -0.012248
1235   -0.012248
1236   -0.012248
1237   -0.012248
Length: 1237, dtype: float64

Portfolio VaR 95%: -0.015328220917102484


Portfolio risk is lower than the risk of the most volatile sectors,
diversification benefits from combined correlated assets